<a href="https://colab.research.google.com/github/rikaahire/cfpb-admin-effects/blob/main/classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

credit_reporting_variants = [
    'Credit reporting',
    'Credit reporting or other personal consumer reports',
    'Credit reporting, credit repair services, or other personal consumer reports'
]

payday_variants = [
    'Payday loan',
    'Payday loan, title loan, or personal loan',
    'Payday loan, title loan, personal loan, or advance loan'
]

bank_variants = [
    'Bank account or service',
    'Checking or savings account'
]

credit_card_variants = [
    'Credit card',
    'Credit card or prepaid card',
    'Prepaid card'
]

money_transfer_variants = [
    'Money transfers',
    'Virtual currency',
    'Money transfer, virtual currency, or money service'
]

company_response_variants = [
    'Closed',
    'Closed with explanation'
]

referral_variants = [
    'Referral',
    'Web Referral'
]

manual_variants = [
    'Email',
    'Fax',
    'Postal mail'
]

needed_cols = [
    'Date received',
    'Date sent to company',
    'Product',
    'Submitted via',
    'Company response to consumer',
    'Timely response?',
    'ZIP code'
]

start_date = pd.to_datetime('2021-01-20')

print("Processing in chunks to save RAM...")

chunks = []
for chunk in pd.read_csv(url,
                         compression='zip',
                         usecols=needed_cols,
                         chunksize=100000,
                         low_memory=False):
    chunk['Product'] = chunk['Product'].replace(credit_reporting_variants, 'Credit Reporting')
    chunk['Product'] = chunk['Product'].replace(payday_variants, 'Payday Loan')
    chunk['Product'] = chunk['Product'].replace(bank_variants, 'Bank Account or Service')
    chunk['Product'] = chunk['Product'].replace(credit_card_variants, 'Credit Card or Prepaid Card')
    chunk['Product'] = chunk['Product'].replace(money_transfer_variants, 'Money Transfer, Virtual Currency, or Money Service')
    chunk['Company response to consumer'] = chunk['Company response to consumer'].replace(company_response_variants, 'Closed with Explanation')
    chunk['Submitted via'] = chunk['Submitted via'].replace(manual_variants, 'Email, Fax, or Postal Mail')
    chunk['Submitted via'] = chunk['Submitted via'].replace(referral_variants, 'Referral')
    chunk['Date received'] = pd.to_datetime(chunk['Date received'])
    filtered_chunk = chunk[chunk['Date received'] >= start_date]

    chunks.append(filtered_chunk)

df = pd.concat(chunks, ignore_index=True)

print(f"Success! Dataframe loaded with {len(df):,} rows.")
print(f"Date range: {df['Date received'].min().date()} to {df['Date received'].max().date()}")

df.head()

Processing in chunks to save RAM...
Success! Dataframe loaded with 13,085,290 rows.
Date range: 2021-01-20 to 2026-05-07


,Date received,Product,ZIP code,Submitted via,Date sent to company,Company response to consumer,Timely response?
0,2024-01-05,Credit Reporting,60502,Web,2024-01-05,Closed with non-monetary relief,Yes
1,2024-01-21,Credit Reporting,27401,Web,2024-01-21,Closed with Explanation,Yes
2,2024-01-23,Credit Reporting,46222,Web,2024-01-23,Closed with Explanation,Yes
3,2024-01-29,Credit Reporting,60411,Web,2024-01-29,Closed with non-monetary relief,Yes
4,2025-04-27,Credit Reporting,06517,Web,2025-04-27,Closed with Explanation,Yes


In [2]:
df.shape

(13085290, 7)

In [3]:
new_names = {'Date received': 'date_received', 'Product': 'product', 'Submitted via':'submitted_via', 'Date sent to company':'date_sent_to_company', 'Company response to consumer':'company_response_to_consumer', 'Timely response?':'timely_response', 'ZIP code':'zip_code'}
df = df.rename(columns=new_names)
df = df[df['company_response_to_consumer'] != 'In progress']
categorical_cols = ['product', 'submitted_via', 'company_response_to_consumer', 'timely_response', 'zip_code']
df = df[~df['zip_code'].astype(str).str.contains('X')]
for col in categorical_cols:
    df[col] = df[col].astype('category')

In [4]:
fraction_similar = (((df['date_received'] == df['date_sent_to_company']).sum())/df.shape[0])
print(f"Similarity between the date the CFPB received the complaint and the date the CFPB sent the complaint to the company: {fraction_similar:.2%}")

Similarity between the date the CFPB received the complaint and the date the CFPB sent the complaint to the company: 96.29%


In [7]:
!pip install census us
from census import Census
from us import states

# Replace with your actual key
c = Census("1e55abc84d703c5a4403570935082050f58fb16a")

# Fetch specific variables for all ZCTAs (ZIP Codes)
# B19013_001E = Median Household Income
# B01003_001E = Total Population
# B15003_022E = Bachelor's Degree
# B03002_003E = % White (non-Hispanic)
# B03002_012E = % Hispanic or Latino
# B01002_001E = Median Age

variables = (
    'NAME',
    'B19013_001E', # Median Income
    'B01003_001E', # Total Pop
    'B15003_022E', # Bachelor's Degree
    'B03002_003E', # White
    'B03002_012E', # Hispanic
    'B02001_003E', # Black
    'B01002_001E',  # Median Age
    'B23025_003E', # Civilian Labor Force
    'B23025_005E', # Unemployed Count
    'B25003_001E', # Total Housing Units
    'B25003_002E'  # Owner Occupied Units
)

data = c.acs5.get(variables, {'for': 'zip code tabulation area:*'})

# Convert to DataFrame
census_df = pd.DataFrame(data)
census_df.rename(columns={
    'B19013_001E': 'median_income',
    'B01003_001E': 'total_population',
    'B15003_022E': 'bachelor_degree',
    'B03002_003E': 'white',
    'B03002_012E': 'hispanic_or_latino',
    'B02001_003E': 'black',
    'B01002_001E': 'median_age',
    'B23025_003E': 'labor_force',
    'B23025_005E': 'unemployed_count',
    'B25003_001E': 'total_units',
    'B25003_002E': 'owner_occupied',
    'zip code tabulation area': 'zip_code'
}, inplace=True)
# 4. Calculate all percentages
# Adding epsilon 1e-6 to avoid division by zero
pop = census_df['total_population'] + 1e-6

census_df['percent_black'] = census_df['black'] / pop
census_df['percent_hispanic'] = census_df['hispanic_or_latino'] / pop
census_df['percent_white'] = census_df['white'] / pop
census_df['percent_bachelors'] = census_df['bachelor_degree'] / pop

census_df['percent_unemployed'] = census_df['unemployed_count'] / (census_df['labor_force'] + 1e-6)
census_df['homeownership_rate'] = census_df['owner_occupied'] / (census_df['total_units'] + 1e-6)

# 5. Final Selection
census_df = census_df[[
    'zip_code', 'median_income', 'total_population',
    'median_age', 'percent_white', 'percent_black',
    'percent_hispanic', 'percent_bachelors', 'percent_unemployed', 'homeownership_rate'
]]

census_df.head()

ConnectionError: HTTPSConnectionPool(host='api.census.gov', port=443): Max retries exceeded with url: /data/2024/acs/acs5?get=NAME%2CB19013_001E%2CB01003_001E%2CB15003_022E%2CB03002_003E%2CB03002_012E%2CB02001_003E%2CB01002_001E%2CB23025_003E%2CB23025_005E%2CB25003_001E%2CB25003_002E&for=zip+code+tabulation+area%3A%2A&key=1e55abc84d703c5a4403570935082050f58fb16a (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7b3079177890>: Failed to establish a new connection: [Errno 101] Network is unreachable'))

In [ ]:
missing_values = df.isnull().sum()

missing_values_percentage = (missing_values / len(df)) * 100

missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage (%)': missing_values_percentage})
missing_data.sort_values(by='Percentage (%)', ascending=False)

In [ ]:
df.dropna(inplace=True)
df.head()

In [ ]:
missing_values = census_df.isnull().sum()

missing_values_percentage = (missing_values / len(census_df)) * 100

missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage (%)': missing_values_percentage})
missing_data.sort_values(by='Percentage (%)', ascending=False)

In [ ]:
# https://cfpb.github.io/api/ccdb/fields.html
df.drop('date_sent_to_company', axis = 1, inplace = True)
df['date_received'] = pd.to_datetime(df['date_received'])
df.head()

In [ ]:
bins = [
    pd.Timestamp('2021-01-20'),
    pd.Timestamp('2025-01-20'),
    pd.Timestamp('2029-01-20')
]

labels = ['Biden', 'Trump_Two']

df['president_date_received'] = pd.cut(df['date_received'], bins=bins, labels=labels, right=False, ordered=False)
df.drop('date_received', axis = 1, inplace = True)

To prevent identifying individuals in sparsely populated areas, the CFPB only publishes the ZIP code if the Census Tabulation Area associated with it has at least 20,000 residents.

If you live in a small town or a rural area with fewer than 20,000 people, the database automatically masks your ZIP code as "XXXXX" to ensure your complaint can't be traced back to your household.

In [ ]:
# Ensure zip codes are strings and formatted the same way (5 digits)
census_df['zip_code'] = census_df['zip_code'].astype(str).str.zfill(5)
df['zip_code'] = df['zip_code'].astype(str).str.zfill(5)

# Merge with your complaints data
# 'left' join preserves all complaints even if a zip isn't in the census file
df = df.merge(census_df, on='zip_code', how='inner')
df.drop(['zip_code'], axis = 1, inplace = True)
df.head()

In [ ]:
df.shape

In [ ]:
df['president_date_received'].value_counts()

In [ ]:
df['product'].value_counts()

In [ ]:
df['submitted_via'].value_counts()

In [ ]:
df['company_response_to_consumer'].value_counts()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.dropna(inplace=True)
print(df.isnull().sum())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

ohe_cols = ['product', 'submitted_via', 'company_response_to_consumer',
       'timely_response']
encoder = OneHotEncoder(sparse_output=True, drop='first')
X_sparse = encoder.fit_transform(df[ohe_cols])

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

# 1. Scale the numerical variables
# We convert them to a 2D array for the scaler
scaler = StandardScaler()
num_cols = ['median_income', 'total_population', 'median_age', 'percent_white', 'percent_black', 'percent_hispanic', 'percent_bachelors', 'percent_unemployed', 'homeownership_rate']
X_scaled_num = scaler.fit_transform(df[num_cols])

# 2. Convert the scaled numbers to a sparse matrix
# This keeps the math consistent for hstack
X_scaled_num_sparse = csr_matrix(X_scaled_num)

# 3. Combine with your One-Hot Encoded features
# X_sparse is your categorical data from your prompt
X_final = hstack([X_scaled_num_sparse, X_sparse])

# 4. Update your feature names list for the Logistic Regression coefficients
feature_names = num_cols + list(encoder.get_feature_names_out(ohe_cols))

print(f"Final feature count: {len(feature_names)}")
print(feature_names)

In [ ]:
y = df['president_date_received']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=5,
    stratify=y
)
import gc
del X_sparse
del X_scaled_num_sparse
del X_final
del df
gc.collect()

In [ ]:
# Standardize the sparse data (Mean 0, Std 1) for correlation
from sklearn.preprocessing import MaxAbsScaler
import numpy as np
sample_size = 20000
indices = np.random.choice(X_train.shape[0], sample_size, replace=False)
X_sample_sparse = X_train[indices]

X_scaled = MaxAbsScaler().fit_transform(X_sample_sparse)

# 4. Sparse multiplication
n_samples = X_sample_sparse.shape[0]
# Use .tocsr() to ensure efficient multiplication
covariance_matrix = (X_scaled.T.tocsr() @ X_scaled.tocsr()) / (n_samples - 1)

# 5. Convert to DataFrame (now safe because we reduced features)
corr_matrix_dense = covariance_matrix.toarray()
corr_df = pd.DataFrame(corr_matrix_dense, columns=feature_names, index=feature_names)

# 6. Unstack and filter
corr_pairs = corr_df.unstack()
highly_correlated_pairs = corr_pairs[
    (corr_pairs >= 0.7) & (corr_pairs < 1.0)
].sort_values(ascending=False)

print(highly_correlated_pairs)

Pearson correlation requires the data to be centered (mean = 0). However, centering sparse data turns every 0 into a non-zero number, which would immediately crash the RAM by making the matrix dense. As a result, while MaxAbsScaler doesn't center the data, the resulting "correlation" matrix is still helpful for identifying redundancy. If two features have a score of 0.9 in this matrix, they are highly collinear, and one should be dropped, even if it's not exactly Pearson correlation.

product_Credit Reporting: Over 70% of all complaints in recent years are about Credit Reporting, so it may be the feature the model depends on to distinguish between administrations since the amount of complaints in this category may have changed over time.

submitted_via_Web: Most people disputing credit report errors do so online. As a result, it may have a similar distribution in the data as product_Credit Reporting but may provide less insight on regulatory enforcement.


timely_response_Yes: While this provides information on company performance, it is something that happens after the complaint is filed so may correlate with how the complaint is submitted or with the product type.

In [ ]:
# https://stackoverflow.com/questions/29294983/how-to-calculate-correlation-between-all-columns-and-remove-highly-correlated-on
# source: https://www.statology.org/cramers-v-in-python/

# Find the index of the column you want to drop
col_to_drop = 'submitted_via_Web'
drop_idx = feature_names.index(col_to_drop)

print(f"Index of '{col_to_drop}': {drop_idx}")

# Create a list of all indices except the one we want to drop
keep_indices = [i for i in range(len(feature_names)) if i != drop_idx]

# Slice the sparse matrices
X_train = X_train[:, keep_indices]
X_test = X_test[:, keep_indices]

# Update your feature names list to keep track of what's left
feature_names = [feature_names[i] for i in keep_indices]

print(f"New shape: {X_train.shape}")

# Find the index of the column you want to drop
col_to_drop = 'timely_response_Yes'
drop_idx = feature_names.index(col_to_drop)

print(f"Index of '{col_to_drop}': {drop_idx}")

# Create a list of all indices except the one we want to drop
keep_indices = [i for i in range(len(feature_names)) if i != drop_idx]

# Slice the sparse matrices
X_train = X_train[:, keep_indices]
X_test = X_test[:, keep_indices]

# Update your feature names list to keep track of what's left
feature_names = [feature_names[i] for i in keep_indices]

print(f"New shape: {X_train.shape}")

In [ ]:
# https://www.geeksforgeeks.org/python/detecting-multicollinearity-with-vif-python/
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

sample_size = min(100000, X_train.shape[0])
indices = np.random.choice(X_train.shape[0], sample_size, replace=False)
X_sample_df = pd.DataFrame(X_train[indices].toarray(), columns=feature_names)
vif_data = add_constant(X_sample_df)

vif_series = pd.DataFrame()
vif_series["Feature"] = vif_data.columns
vif_series["VIF"] = [variance_inflation_factor(vif_data.values, i)
                     for i in range(vif_data.shape[1])]

print(vif_series[vif_series['Feature'] != 'const'].sort_values(by="VIF", ascending=False))

In [ ]:
# https://stackoverflow.com/questions/29294983/how-to-calculate-correlation-between-all-columns-and-remove-highly-correlated-on
# source: https://www.statology.org/cramers-v-in-python/

# Find the index of the column you want to drop
col_to_drop = 'percent_white'
drop_idx = feature_names.index(col_to_drop)

print(f"Index of '{col_to_drop}': {drop_idx}")

# Create a list of all indices except the one we want to drop
keep_indices = [i for i in range(len(feature_names)) if i != drop_idx]

# Slice the sparse matrices
X_train = X_train[:, keep_indices]
X_test = X_test[:, keep_indices]

# Update your feature names list to keep track of what's left
feature_names = [feature_names[i] for i in keep_indices]

print(f"New shape: {X_train.shape}")

VIF < 5: very little multicollinearity, and model coefficients should be reliable


VIF 5–10: Moderate correlation, but usually acceptable for most models


VIF > 10: a lot of multicollinearity


product_Credit Reporting has the highest VIF (6.6) since it is the most seen category in the dataset. Since we are using Logistic Regression, we want to keep features that have a strong relationship with the target. Credit Reporting might be the most important predictor. If we drop it, our model's accuracy might drop since this feature might be the one with the most information for the model.

To lower the VIF and make your model stable, you should drop one of the three major racial groups. This acts as your "reference" or "baseline" category.

Drop percent_white so percent_white is the reference category. Since in US political modeling the standard practice is to use the majority population as the reference and then understand how changes in the minority populations change predictions.

In [ ]:
sample_size = min(100000, X_train.shape[0])
indices = np.random.choice(X_train.shape[0], sample_size, replace=False)
X_sample_df = pd.DataFrame(X_train[indices].toarray(), columns=feature_names)
vif_data = add_constant(X_sample_df)

vif_series = pd.DataFrame()
vif_series["Feature"] = vif_data.columns
vif_series["VIF"] = [variance_inflation_factor(vif_data.values, i)
                     for i in range(vif_data.shape[1])]

print(vif_series[vif_series['Feature'] != 'const'].sort_values(by="VIF", ascending=False))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, balanced_accuracy_score
import matplotlib.pyplot as plt

lr = LogisticRegression(class_weight='balanced', max_iter=10000, random_state=5).fit(X_train, y_train)
predictions = lr.predict(X_test)
cm = confusion_matrix(y_test, predictions, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot()
plt.title('Confusion Matrix for Test Data')
plt.show()
print('Test Set Balanced Accuracy:', balanced_accuracy_score(y_test, predictions))

In [ ]:
# 1. Create the DataFrame using only the first (and only) row of coefficients
# We use lr.classes_[1] because the coefficients usually correspond to the "positive" class (class 1)
coef_df = pd.DataFrame(
    lr.coef_[0],
    index=feature_names,
    columns=['Coefficient']
)

# 2. Add an Odds Ratio column for easier interpretation
coef_df['Odds_Ratio'] = np.exp(coef_df['Coefficient'])

# 3. Sort to see which features favor which administration
# Higher Odds Ratio (>1) favors the "Positive" class (usually lr.classes_[1])
# Lower Odds Ratio (<1) favors the "Negative" class (usually lr.classes_[0])
coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

print(f"Model classes: {lr.classes_}")
print(f"Positive coefficients favor: {lr.classes_[1]}")
print(f"Negative coefficients favor: {lr.classes_[0]}")

display(coef_df)

Positive Coefficients / Odds Ratio > 1: As this feature increases, the complaint is more likely to be from the Trump_Two era.

Negative Coefficients / Odds Ratio < 1: As this feature increases, the complaint is more likely to be from the Biden era.

If median_income has an Odds Ratio of 1.15, it means that for every 1-unit increase in standardized income, the odds of the complaint belonging to the Trump era increase by 15%.If product_Student loan has an Odds Ratio of 0.60, it means that feature is associated with the Biden era (since $0.60 < 1.0$).

The strongest predictor for the Trump_Two administration is that the company responded to the consumer complaint in an untimely manner. Untimely response means that the company did not respond to the consumer complaint on time. Based on the model's odd ratio, a complaint is 4.6 times more likely to be from the current Trump administration if the company did not respond on time. This could imply a change in regulatory pressure for companies to respond to consumers or a shift in how companies regard the CFPB's deadlines.

In terms of products, credit reporting and debt are some of the model's top predictors. A complaint about credit reporting is 2.4 times as likely in the current administration and a complaint about debt management is 1.71 times as likely in the current administration.

In terms of demographic features, complaints from a ZIP code with a higher percentage of Hispanics or Latinos and/or a ZIP code with higher rates of homeownership are more likely to be during the Trump administration.

On the other hand, the features that have almost zero predictive power for the model are median income and median age. This means that complaints are more driven by the product rather than the age or income of a person.

The main changes between administrations is characterized by a drop in company responsiveness (corresponding to the higher probability of seeing an untimely response to consumer complaints), a change in the product that is the focus of consumer complaints (products during the Biden administration tended to be more interest related, such as mortgage and student loan, while products during the Trump administration tended to be more credit driven, such as credit reporting and debt/credit management), and a change in how consumers interact with the CFPB (Biden's administration saw more complaints submitted via referral and via phone compared to the Trump administration).

Choice for Logistic Regression:
- Fast to train
- Better results than Decision Tree classifier, which overpredicts second Trump administration for consumer complaints during the Biden administration
- almost identical accuracy as LinearSVC, except Logistic Regression results are more interpretable since it is designed for Odds Ratios

The model is only performing about 6% better than random chance.

The data suggests that there is a lack of distinct patterns in the features.

Consumers tend to complain about the same products regardless of the administration. Credit reporting consists of 77% of all complaints. As a result, most complaint's product feature does not help the model distinguish between administrations.

Similarly, most complaints are submitted through the Web. Since the behavior of how people submit complaints does not change a lot when the administration changes, this feature does not have a lot of predictive power.

The ways companies respond appear to be standard industry practices that do not change based on who is in the White House.

As a result, the model does not perform much higher than random since the features do not capture any political or economic changes that define an administration.

To improve it might help to look at the Complaint Narrative feature to find specific policy-related language that changes between the administrations (this is done in the Topic Modeling section). It might also help to examine if consumer sentiment is a strong predictor of administration (this is done in the Sentiment and Severity Analysis section).

In [31]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Verify: Biden might be 0 and Trump_Two might be 1
print(f"Classes: {le.classes_}")

Classes: ['Biden' 'Trump_Two']


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, ConfusionMatrixDisplay

# Initialize the model
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',       # Standard for hidden layers
    solver='adam',           # Best for large datasets
    alpha=0.0001,            # L2 penalty to prevent overfitting
    batch_size=1024,         # Large batches for 11M rows
    learning_rate_init=0.01,
    max_iter=10,             # We use early stopping instead of many epochs
    early_stopping=True,     # CRITICAL: Stops training when validation score stalls
    validation_fraction=0.1, # Uses 10% of data to check for overfitting
    verbose=True,            # Shows progress so you know it hasn't crashed
    random_state=5
)

# Train the model
mlp.fit(X_train, y_train_encoded)

# Evaluate
predictions = mlp.predict(X_test)
print('Test Set Balanced Accuracy:', balanced_accuracy_score(y_test_encoded, predictions))

# 3. Plot Confusion Matrix
cm = confusion_matrix(y_test_encoded, predictions, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot()
plt.title("Confusion Matrix for Test Data (Neural Network)")
plt.show()

Iteration 1, loss = 0.66849173
Validation score: 0.594239
Iteration 2, loss = 0.66726928
Validation score: 0.593628
Iteration 3, loss = 0.66673281
Validation score: 0.594884
Iteration 4, loss = 0.66630729
Validation score: 0.594978
Iteration 5, loss = 0.66598952
Validation score: 0.595096
Iteration 6, loss = 0.66570304
Validation score: 0.595399
Iteration 7, loss = 0.66546009
Validation score: 0.595840
Iteration 8, loss = 0.66522904
Validation score: 0.595420
Iteration 9, loss = 0.66505534
Validation score: 0.596515
Iteration 10, loss = 0.66487218
Validation score: 0.596327


In [ ]:
from sklearn.inspection import permutation_importance

# 1. Run the importance check on your test set
result = permutation_importance(
    mlp, X_test, y_test_encoded,
    n_repeats=10,        # Scramble each feature 10 times to be sure
    random_state=5,
    n_jobs=-1            # Use all CPU cores
)

# 2. Display the results
import pandas as pd
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance_Mean': result.importances_mean,
    'Importance_Std': result.importances_std
}).sort_values(by='Importance_Mean', ascending=False)

print(importance_df)

In [ ]:
# !pip install shap
import shap

# 1. Use a small sample (SHAP is slow on 11M rows)
X_sample = X_test[:1000]

# 2. Initialize the explainer
explainer = shap.KernelExplainer(mlp.predict_proba, X_sample)
shap_values = explainer.shap_values(X_sample)

# 3. Visualize the "Impact"
# This shows how much each feature pushes the result left or right
shap.summary_plot(shap_values[1], X_sample, feature_names=feature_names)

In [ ]:

# Would want more information on what features your using, and why you selected these features

# Why might it be so hard to predict admin? May be worth investigating given there are some features that are correlated with admin, e.g., consumer loans, submitted via web

# Would be interesting to compare performance vs. non-explainable models such as neural networks, etc.